# 235081 M.Hamdan AI Project

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

# 1. Load the new static dataset
# The delimiter is '|' for this specific file
df = pd.read_csv('data.csv', sep='|')

# 2. Data Cleaning
# Remove non-numeric identifiers so the AI only learns from technical headers
# 'legitimate' is the target (0 = Malware, 1 = Safe)
X = df.drop(['Name', 'md5', 'legitimate'], axis=1) 
y = df['legitimate']

# 3. Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Train the Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. Save the new model and the specific feature list
joblib.dump(model, 'malware_detector.pkl')
joblib.dump(X.columns.tolist(), 'model_features.pkl')

print("Static Model and Features Saved!")

: 

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

# Generate predictions on the test set
predictions = model.predict(X_test)

# Print the numeric accuracy percentage
print(f"Model Accuracy: {accuracy_score(y_test, predictions) * 100:.2f}%")

# Print a detailed report (Precision, Recall, F1-Score)
print("\nClassification Report:\n", classification_report(y_test, predictions))

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import pefile
import joblib
import pandas as pd
import os

# 1. UI Setup
uploader = widgets.FileUpload(accept='.exe,.com', multiple=False)
out = widgets.Output()

def on_file_upload(change):
    with out:
        clear_output()
        if not uploader.value:
            return
            
        uploaded_file = uploader.value[0]
        file_name = uploaded_file['name']
        file_content = bytes(uploaded_file['content']) # Fix for memoryview
        
        print(f" Analyzing: {file_name}")
        
        # --- LAYER 1: EICAR SIGNATURE CHECK ---
        # We check for the specific text string used in the EICAR test file
        if b"X5O!P%@AP[4" in file_content:
            print("\n" + "!"*40)
            print(" VERDICT: EICAR ANTI-VIRUS TEST FILE DETECTED")
            print("Note: This is a safe string used to test security logic.")
            print("!"*40)
            return

        # --- LAYER 2: PE STRUCTURE CHECK & AI PREDICTION ---
        try:
            # Load the model and features
            if not os.path.exists('malware_detector.pkl'):
                print(" Error: 'malware_detector.pkl' not found. Run the training cell first!")
                return
                
            model = joblib.load('malware_detector.pkl')
            expected_features = joblib.load('model_features.pkl')

            # Use pefile to parse the headers
            pe = pefile.PE(data=file_content)
            
            # Extract features to match your Kaggle dataset
            feature_data = {}
            for feat in expected_features:
                # Dynamically find headers (Machine, SizeOfStackReserve, etc.)
                val = getattr(pe.FILE_HEADER, feat, None)
                if val is None:
                    val = getattr(pe.OPTIONAL_HEADER, feat, 0)
                feature_data[feat] = val

            # Format for the model
            final_df = pd.DataFrame([feature_data])[expected_features]

            # Make the Verdict
            prediction = model.predict(final_df)
            
            print("\n" + "="*30)
            if prediction[0] == 1:
                print(" VERDICT: MALICIOUS (MALWARE)")
                print("The AI found patterns matching known threats.")
            else:
                print(" VERDICT: BENIGN (SAFE)")
                print("The file appears to be a normal program.")
            print("="*30)
                
        except pefile.PEFormatError:
            print(" Error: Not a valid Windows Executable (No MZ Header found).")
        except Exception as e:
            print(f" ERROR: {e}")

# Link button to function
uploader.observe(on_file_upload, names='value')
print("System Ready. Upload an .exe or EICAR file to test.")
display(uploader, out)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

# Generate the confusion matrix
y_pred = model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

# Plot it
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Safe', 'Malware'], 
            yticklabels=['Safe', 'Malware'])
plt.xlabel('Predicted Label')
plt.ylabel('Actual Label')
plt.title('Malware Detection Accuracy')
plt.show()

In [ ]:
%pip install ipywidgets

In [ ]:
import os

# Create a valid-looking PE stub
with open("demo_malware.exe", "wb") as f:
    # 1. DOS Header starts with 'MZ'
    f.write(b"MZ" + b"\x00" * 58) 
    
    # 2. At offset 0x3C (60 bytes), we must point to the PE header 
    # Let's say the PE header starts at 128 bytes (0x80)
    f.write(b"\x80\x00\x00\x00") 
    
    # 3. Fill the gap up to 128 bytes
    f.write(b"\x00" * (128 - f.tell()))
    
    # 4. The PE Signature (This is what pefile looks for)
    f.write(b"PE\x00\x00")
    
    # 5. Add some random data so it's not empty
    f.write(os.urandom(1024))

print("Valid PE stub created as demo_malware.exe")